### Gráfica 1: Distribución en Estado Estacionario (Ley de Potencias)
La presente sección analiza la distribución de tamaños de los grupos en el estado estacionario del sistema para un tamaño poblacional de $N = 10,000$ y una tasa de fragmentación de $\nu = 0.01$.

**Análisis de la Gráfica:**
Al representar la frecuencia relativa normalizada respecto al grupo base ($n_s/n_1$) en función de la fuerza de ataque ($s$) en escala log-log, se observa una excelente concordancia con el comportamiento teórico de ley de potencias caracterizado por el exponente universal $\alpha = 2.5$. Este resultado valida empíricamente que la simulación reproduce de manera robusta la estructura de escala esperada en conflictos asimétricos, excluyendo los efectos de borde en la cola superior y la inyección artificial en $s=1$.

In [ ]:
N_val = 10_000
nu_val = 0.01
pasos_val = 1_000_000

resultado = simular2(N=N_val, nu=nu_val, pasos=pasos_val, semilla=42)
distribucion = resultado["distribucion_tamanos"]

s = np.array(sorted(distribucion.keys()))
ns = np.array([distribucion[int(val)]["count"] for val in s])

plt.figure(figsize=(9, 6))

indice_unidades = np.where(s == 1)[0]
if len(indice_unidades) > 0:
    n1 = ns[indice_unidades[0]]
    if n1 > 0:
        ns_norm = ns / n1
        plt.loglog(s, ns_norm, 'bo', alpha=0.7, label='Datos de la simulación ($n_s/n_1$)')
        
        s_max = max(s)
        limite_superior = max(5, s_max // 2 if s_max < 30 else s_max // 3)
        mask = (s >= 2) & (s <= limite_superior) & (ns_norm > 0)
        
        if np.sum(mask) >= 2:
            log_s = np.log(s[mask])
            log_ns_n1 = np.log(ns_norm[mask])
            pendiente, intercepto = np.polyfit(log_s, log_ns_n1, 1)
            alpha_estimado = -pendiente
            
            s_fit = np.linspace(2, limite_superior, 100)
            ns_fit = np.exp(intercepto) * (s_fit ** pendiente)
            plt.loglog(s_fit, ns_fit, 'g-', linewidth=2, 
                     label=f'Tendencia ajustada ($\alpha_{{fit}} = {alpha_estimado:.2f}$)')

s_ref = np.linspace(2, max(s) // 2, 100)
plt.loglog(s_ref, s_ref**(-2.5), 'k--', linewidth=2, label='Valor teórico de referencia ($\alpha = 2.5$)')

plt.xlabel('Fuerza de ataque / tamaño del grupo ($s$)', fontsize=12)
plt.ylabel('$n_s / n_1$', fontsize=12)
plt.title(f'Distribución normalizada en Estado Estacionario ($N={N_val}, \nu={nu_val}$)', fontsize=13)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

### Gráfica 2: Sensibilidad no lineal de $\alpha$ frente a $\nu$ (Escala Semi-Log)
Se evalúa la estabilidad del sistema mediante un barrido en el parámetro de fragmentación $\nu$ utilizando múltiples semillas y un número adaptativo de iteraciones para garantizar la convergencia al estado estacionario.

**Análisis del Comportamiento:**
Utilizando una escala logarítmica en el eje horizontal para distribuir adecuadamente los órdenes de magnitud de $\nu$, la tendencia polinómica revela un comportamiento no lineal. El sistema muestra una alta sensibilidad en regímenes de baja fragmentación, identificando el entorno de $\nu \approx 0.005$ a $0.01$ como la región óptima donde el exponente se alinea estrechamente con el valor teórico universal de $\alpha = 2.5$.

In [ ]:
valores_nu = [0.001, 0.002, 0.004, 0.006, 0.008, 0.01, 0.015, 0.02, 0.03, 0.05, 0.08, 0.1]
semillas = [42, 101, 202]
resultados_nu = {}

def estimar_alpha_robusto(valores_s, conteos):
    s_max = max(valores_s)
    limite_superior = max(5, s_max // 2 if s_max < 30 else s_max // 3)
    mask = (valores_s >= 2) & (valores_s <= limite_superior) & (conteos > 0)
    
    if np.sum(mask) < 2:
        mask = (valores_s > 0) & (conteos > 0)
        
    log_s = np.log(valores_s[mask])
    log_ns = np.log(conteos[mask])
    
    pendiente, _ = np.polyfit(log_s, log_ns, 1)
    return -pendiente

for nu_v in valores_nu:
    alphas_semillas = []
    pasos_adaptativos = 1_500_000 if nu_v <= 0.002 else (800_000 if nu_v <= 0.008 else 400_000)
    
    for sem in semillas:
        res = simular2(N=5000, nu=nu_v, pasos=pasos_adaptativos, semilla=sem)
        tamanos = res["tamanos_grupos"]
        vals_s, conts = np.unique(tamanos, return_counts=True)
        a_est = estimar_alpha_robusto(vals_s, conts)
        
        if not np.isnan(a_est) and 0.5 < a_est < 5.0:
            alphas_semillas.append(a_est)
            
    resultados_nu[nu_v] = {
        "media": np.mean(alphas_semillas) if alphas_semillas else np.nan,
        "std": np.std(alphas_semillas) if alphas_semillas else np.nan
    }

nus_validos = [nu for nu in valores_nu if not np.isnan(resultados_nu[nu]["media"])]
medias_alpha = [resultados_nu[nu]["media"] for nu in nus_validos]
stds_alpha = [resultados_nu[nu]["std"] for nu in nus_validos]

log_nus = np.log(nus_validos)
coefs = np.polyfit(log_nus, medias_alpha, 2)
nu_fino = np.logspace(np.log10(min(nus_validos)), np.log10(max(nus_validos)), 200)
tendencia_suave = np.polyval(coefs, np.log(nu_fino))

plt.figure(figsize=(9, 5))
plt.errorbar(nus_validos, medias_alpha, yerr=stds_alpha, fmt='o', color='darkgreen', ecolor='gray', 
             capsize=4, elinewidth=1.2, markersize=6, label='$\alpha$ estimado (Media ± Desv. Est.)')

plt.plot(nu_fino, tendencia_suave, '-', color='tab:blue', linewidth=2, label='Tendencia no lineal (Ajuste)')
plt.axhline(y=2.5, color='r', linestyle='--', label='Valor de referencia universal ($\alpha = 2.5$)')

plt.xscale('log')
plt.xlabel('Parámetro de fragmentación $\nu$ (Escala Log)', fontsize=12)
plt.ylabel('Exponente $\alpha$ estimado', fontsize=12)
plt.title('Sensibilidad no lineal del exponente $\alpha$ frente a $\nu$', fontsize=13)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

### Gráfica 3: Sensibilidad de $\alpha$ frente a $\nu$ con Tendencia Lineal (Escala Log-Log)
Esta perspectiva transforma ambos ejes a escala logarítmica para contrastar si la relación entre el exponente crítico $\alpha$ y la tasa de fragmentación $\nu$ responde a una ley de potencias directa.

**Análisis del Comportamiento:**
El ajuste de regresión lineal sobre el espacio log-log arroja una pendiente cercana a cero ($m \approx -0.040$). Desde el punto de vista del informe, esto demuestra cuantitativamente que no existe una dependencia de ley de potencias fuerte entre ambas magnitudes; por el contrario, el exponente $\alpha$ muestra una estabilidad considerable y acotada ante variaciones amplias en la intensidad de fragmentación del entorno.

In [ ]:
log_nus = np.log(nus_validos)
log_alphas = np.log(medias_alpha)
m, b = np.polyfit(log_nus, log_alphas, 1)

nu_fino = np.logspace(np.log10(min(nus_validos)), np.log10(max(nus_validos)), 200)
alpha_ajustado = np.exp(b) * (nu_fino ** m)

plt.figure(figsize=(9, 5))
plt.errorbar(nus_validos, medias_alpha, yerr=stds_alpha, fmt='o', color='darkgreen', ecolor='gray', 
             capsize=4, elinewidth=1.2, markersize=6, label='$\alpha$ estimado (Media ± Desv. Est.)')

plt.plot(nu_fino, alpha_ajustado, '-', color='tab:blue', linewidth=2, 
         label=f'Ajuste lineal log-log (Pendiente m = {m:.3f})')

plt.axhline(y=2.5, color='r', linestyle='--', label='Valor de referencia universal ($\alpha = 2.5$)')

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Parámetro de fragmentación $\nu$ (Escala Log)', fontsize=12)
plt.ylabel('Exponente $\alpha$ estimado (Escala Log)', fontsize=12)
plt.title('Sensibilidad de $\alpha$ frente a $\nu$ con Tendencia Lineal (Log-Log)', fontsize=13)
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

### Gráfica 4: Sensibilidad de $\alpha$ frente a $\nu$ (Escala Lineal Estándar)
Se presenta el comportamiento de la sensibilidad utilizando una representación cartesiana convencional con escalas lineales tanto en el eje horizontal como en el vertical.

**Análisis del Comportamiento:**
Esta representación ilustra de forma clara el efecto de compresión numérica en los valores reducidos de $\nu$, donde los puntos iniciales se agrupan densamente en el margen izquierdo. A pesar de la limitación visual en los órdenes menores comparado con las vistas logarítmicas, el ajuste polinómico corrobora la consistencia global de la media y la desviación estándar obtenida a través de múltiples ejecuciones estocásticas.

In [ ]:
coefs_lin = np.polyfit(nus_validos, medias_alpha, 2)
nu_fino_lin = np.linspace(min(nus_validos), max(nus_validos), 200)
tendencia_lin = np.polyval(coefs_lin, nu_fino_lin)

plt.figure(figsize=(9, 5))
plt.errorbar(nus_validos, medias_alpha, yerr=stds_alpha, fmt='o', color='darkgreen', ecolor='gray', 
             capsize=4, elinewidth=1.2, markersize=6, label='$\alpha$ estimado (Media ± Desv. Est.)')

plt.plot(nu_fino_lin, tendencia_lin, '-', color='tab:blue', linewidth=2, label='Tendencia polinómica')
plt.axhline(y=2.5, color='r', linestyle='--', label='Valor de referencia universal ($\alpha = 2.5$)')

plt.xlabel('Parámetro de fragmentación ($\nu$)', fontsize=12)
plt.ylabel('Exponente $\alpha$ estimado', fontsize=12)
plt.title('Sensibilidad de $\alpha$ frente a $\nu$ (Escala Lineal Estándar)', fontsize=13)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()